In [7]:
from textwrap import dedent
from agno.agent import Agent
from agno.models.groq import Groq
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.newspaper4k import Newspaper4kTools
from datetime import datetime
import openai
import requests
import re
from typing import List, Dict, Union
from enum import Enum

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

# Retrieve API keys
groq_api_key = os.getenv('GROQ_API_KEY')
phi_api_key = os.getenv('PHI_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')

print("API keys have been set!")

API keys have been set!


In [5]:

# Check if the API keys are set
print("Groq API key set:" if os.getenv("GROQ_API_KEY") else "Groq API key not set")
print("PHI API key set:" if os.getenv("PHI_API_KEY") else "PHI API key not set")
print("OpenAI API key set:" if os.getenv("OPENAI_API_KEY") else "OpenAI API key not set")


Groq API key set:
PHI API key set:
OpenAI API key set:


### Research Agent (Web Search & Newspaper Search)

In [10]:
# user query
user_query = "Analyze the current updates on Chittagong Port including operational trends, policy changes, and logistical disruptions."

# Function to extract port details from the query using regex.
def extract_port(query: str) -> str:
    # The regex pattern here is simplistic and assumes that port names follow patterns like "Port of ..."
    # You can adjust this pattern for more robust matching or use an NLP library for improved accuracy.
    pattern = r"(Port of [A-Za-z\s&-]+)"
    match = re.search(pattern, query)
    if match:
        return match.group(1).strip()
    return "Port of New York and New Jersey"  # Default port if none detected

# Extract the target port from the query
target_port = extract_port(user_query)

# Define the dynamic research agent for supply chain operations at any port
research_agent = Agent(
    model=Groq(id="llama3-70b-8192"),
    tools=[DuckDuckGoTools(), Newspaper4kTools()],
    description=dedent(f"""\
        You are a Supply Chain Analyst specialized in monitoring real-time operational updates and trends 
        at high-traffic ports globally. Your current focus is on analyzing operations at {target_port}.
        Your expertise includes:

        - Deep investigative research on supply chain operations,
        - Fact-checking and source verification,
        - Data-driven reporting on current logistics trends,
        - Expert analysis of port operations impacting supply chains,
        - Trend analysis and implications for near-future operations,
        - Simplification of complex logistics and operational concepts,
        - Ensuring balanced and ethical perspectives,
        - Integrating global context to supply chain dynamics.
    """),
    instructions=dedent(f"""\
        1. Research Phase
           - Conduct web searches and gather the latest news and updates on port operations globally, then narrow down to {target_port}.
           - Prioritize real-time, authoritative, and non-historic sources.
           - Focus on current logistical challenges, operational disruptions, and policy changes that directly impact supply chain efficiency at {target_port}.

        2. Analysis Phase
           - Extract critical data points regarding current operations, policy adjustments, and logistical updates for {target_port}.
           - Analyze trends in real-time supply chain disruptions or operational improvements.
           - Assess the immediate impact of geopolitical factors and market conditions on port throughput and overall logistics flow.

        3. Writing Phase
           - Develop a compelling headline that reflects the latest operational updates at {target_port}.
           - Structure the report with clear sections focused on current operations:
             - Executive Summary, Critical Updates, Impact Analysis, and Future Outlook.
           - Include relevant quotes, statistics, and near-term implications.

        4. Quality Control
           - Verify all facts with cross-reference from multiple reputable sources.
           - Emphasize current operational details without including extensive historical or unrelated background.
           - Provide clear context for all data and offer actionable insights on operational trends.
    """),
    expected_output=dedent(f"""\
        # {{Compelling Headline on Current Port Operations and Supply Chain Impact}}

        ## Executive Summary
        {{Concise overview of the latest findings regarding operations at {target_port}.}}

        ## Critical Updates
        {{Recent operational changes and logistical disruptions at {target_port} that directly impact supply chains.}}
        {{Breaking news on policy changes and immediate adjustments by port authorities.}}
        {{Statistical evidence on port throughput and current operational trends.}}

        ## Impact Analysis
        {{Immediate implications on supply chain operations and logistics.}}
        {{Stakeholder perspectives including port authorities and key industry players.}}
        {{Analysis of market, geopolitical, and economic influences.}}

        ## Future Outlook
        {{Emerging trends and upcoming changes in port operations at {target_port}.}}
        {{Expert predictions on near-future performance and policy modifications.}}
        {{Potential challenges and actionable opportunities for supply chain optimization.}}

        ## Expert Insights
        {{Notable quotes and analysis from port operation experts and supply chain analysts.}}
        {{Discussion on contrasting viewpoints regarding current operational shifts.}}

        ## Sources & Methodology
        {{List of primary sources with key contributions.}}
        {{Overview of research methodology and data verification for current updates.}}
        
        ---
        Research conducted by Supply Chain Analyst  
        Logistics Analysis Report  
        Published: {{current_date}}  
        Last Updated: {{current_time}}
    """),
    markdown=True,
    show_tool_calls=True,
    add_datetime_to_instructions=True,
)

if __name__ == "__main__":
    # Use the user's query to analyze the updates for the detected port details
    research_agent.print_response(user_query, stream=True)

Output()

### Router Agent

In [11]:
class QueryType(Enum):
    WEB_NEWS = "web_news"          # Web/News Agent only
    RAG = "rag"                    # RAG only for simple queries
    HYBRID = "hybrid"              # Both RAG and Web/News

class RouterAgent:
    def __init__(self, debug: bool = False):
        """Initialize Router Agent with three routing pathways"""
        self.debug = debug
        load_dotenv()
        
        if not os.getenv('GROQ_API_KEY'):
            raise ValueError("GROQ_API_KEY environment variable not set")

        # Router Agent with optimized prompt
        self.router = Agent(
            model=Groq(id="llama3-70b-8192"),
            tools=[DuckDuckGoTools(), Newspaper4kTools()],
            description=dedent("""\
                You are a sophisticated Router Agent specializing in supply chain query analysis.
                Your task is to classify queries into one of three pathways for optimal processing:
                - WEB_NEWS: Real-time data from web or news sources
                - RAG: Simple queries answerable from internal knowledge
                - HYBRID: Complex queries needing both internal knowledge and external data
            """),
            instructions=dedent("""\
                1. Analyze the query to determine its intent, complexity, and data needs.
                2. Classify it into ONE of these categories:
                   - WEB_NEWS: Needs fresh external data (e.g., current rates, news events)
                   - RAG: Simple, general knowledge (e.g., definitions, basic concepts)
                   - HYBRID: Requires both internal insights and external updates
                3. Return ONLY the category name (e.g., "WEB_NEWS", "RAG", "HYBRID").
                4. Avoid over-classification; select the single most appropriate pathway.
            """),
            markdown=True,
            show_tool_calls=True
        )

        # Web/News Retrieval Agent
        self.web_news_agent = Agent(
            model=Groq(id="llama3-70b-8192"),
            tools=[DuckDuckGoTools(), Newspaper4kTools()],
            description=dedent("""\
                Web/News Retrieval Agent for real-time supply chain data.
            """),
            instructions=dedent("""\
                1. Fetch current web and news data relevant to the query.
                2. Use DuckDuckGo for web searches and Newspaper4k for news.
                3. Return raw data structured as:
                   - Web: {web_data}
                   - News: {news_data}
            """),
            markdown=True,
            show_tool_calls=True
        )

        # RAG Agent (Internal knowledge simulation)
        self.rag_agent = Agent(
            model=Groq(id="llama3-70b-8192"),
            tools=[],
            description=dedent("""\
                RAG Agent for simple supply chain queries using internal knowledge.
            """),
            instructions=dedent("""\
                1. Answer using general supply chain knowledge.
                2. Do not perform external searches.
                3. Provide concise, factual responses.
            """),
            markdown=True,
            show_tool_calls=True
        )

        # Analytical Expert Agent (for Hybrid)
        self.analytical_expert = Agent(
            model=Groq(id="llama3-70b-8192"),
            tools=[DuckDuckGoTools(), Newspaper4kTools()],
            description=dedent("""\
                Analytical Expert Agent for combining RAG and Web/News data.
            """),
            instructions=dedent("""\
                1. Combine RAG knowledge with Web/News data.
                2. Provide a structured analysis:
                   - Key Findings
                   - Implications
                   - Recommendations
                3. Ensure clarity and actionable insights.
            """),
            markdown=True,
            show_tool_calls=True
        )

    def _log(self, message: str) -> None:
        if self.debug:
            current_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
            print(f"[{current_time}] [DEBUG] {message}")

    def classify_query(self, query: str) -> List[QueryType]:
        """Classify query with error handling"""
        self._log(f"Classifying query: {query}")
        
        classification_prompt = dedent(f"""\
            You are an expert in supply chain query classification.
            Analyze this query: "{query}"
            
            Classify it into ONE of these pathways:
            - WEB_NEWS: Requires real-time external data (e.g., shipping rates, news updates)
            - RAG: Simple query answerable from general knowledge (e.g., definitions)
            - HYBRID: Complex query needing both internal knowledge and external data
            
            Return ONLY the category name (e.g., "WEB_NEWS", "RAG", "HYBRID").
            Focus on intent and data requirements for accuracy.
        """)
        
        response = self.router.print_response(classification_prompt, stream=False)
        
        # Handle None response
        if response is None:
            self._log("Received None response from router; defaulting to HYBRID")
            return [QueryType.HYBRID]
        
        try:
            category = response.strip().lower()
            query_type = QueryType(category)
            self._log(f"Classified as: {query_type.value}")
            return [query_type]
        except ValueError:
            self._log(f"Invalid category received: {response}; defaulting to HYBRID")
            return [QueryType.HYBRID]

    def route_query(self, query: str) -> Dict[str, Union[str, List[str], dict]]:
        """Route query through appropriate pathways"""
        timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        query_types = self.classify_query(query)
        retrieved_data = {}
        final_response = {}

        for query_type in query_types:
            self._log(f"Processing {query_type.value}")
            
            if query_type == QueryType.WEB_NEWS:
                retrieved_data["web_news"] = self.web_news_agent.print_response(
                    f"Retrieve real-time web and news data for: {query}", stream=False
                )
                final_response["result"] = retrieved_data["web_news"]

            elif query_type == QueryType.RAG:
                final_response["result"] = self.rag_agent.print_response(
                    f"Answer using internal supply chain knowledge: {query}", stream=False
                )

            elif query_type == QueryType.HYBRID:
                retrieved_data["web_news"] = self.web_news_agent.print_response(
                    f"Retrieve real-time web and news data for: {query}", stream=False
                )
                retrieved_data["rag"] = self.rag_agent.print_response(
                    f"Provide general knowledge for: {query}", stream=False
                )
                analysis_prompt = dedent(f"""\
                    Analyze this query: "{query}"
                    Using:
                    - Web/News Data: {retrieved_data['web_news'] or 'No web/news data available'}
                    - Internal Knowledge: {retrieved_data['rag'] or 'No RAG data available'}
                    
                    Provide a structured response:
                    - Key Findings
                    - Implications
                    - Recommendations
                """)
                final_response["result"] = self.analytical_expert.print_response(
                    analysis_prompt, stream=False
                )

        return {
            "query": query,
            "timestamp": timestamp,
            "classification": [qt.value for qt in query_types],
            "retrieved_data": retrieved_data,
            "final_response": final_response,
            "status": "success"
        }

if __name__ == "__main__":
    agent = RouterAgent(debug=True)
    result = agent.route_query("Test query: Current shipping rates from Chittagong Port and how much time it can take to reach New York?")
    print(result)

[2025-02-20 20:01:26] [DEBUG] Classifying query: Test query: Current shipping rates from Chittagong Port and how much time it can take to reach New York?


Output()

[2025-02-20 20:01:27] [DEBUG] Received None response from router; defaulting to HYBRID
[2025-02-20 20:01:27] [DEBUG] Processing hybrid


Output()

ERROR    https://duckduckgo.com?q=shipping+news+from+Chittagong+Port return None. params={'q': 'shipping news from 
         Chittagong Port'} content=None data=None                                                                  
         Traceback (most recent call last):                                                                        
           File "e:\agentic_ai_rag_system\.venv\Lib\site-packages\agno\tools\function.py", line 359, in execute    
             self.result = self.function.entrypoint(**entrypoint_args, **self.arguments)                           
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                           
           File "e:\agentic_ai_rag_system\.venv\Lib\site-packages\pydantic\_internal\_validate_call.py", line 38,  
         in wrapper_function                                                                                       
             return wrapper(*args, **kwargs)                                                                       
                    ^^^^^^^^^^^^^^^^^^^^^^^^                                                                       
           File "e:\agentic_ai_rag_system\.venv\Lib\site-packages\pydantic\_internal\_validate_call.py", line 111, 
         in __call__                                                                                               
             res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))             
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^             
           File "e:\agentic_ai_rag_system\.venv\Lib\site-packages\agno\tools\duckduckgo.py", line 88, in           
         duckduckgo_news                                                                                           
             return json.dumps(ddgs.news(keywords=query, max_results=(self.fixed_max_results or max_results)),     
         indent=2)                                                                                                 
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^      
           File "e:\agentic_ai_rag_system\.venv\Lib\site-packages\duckduckgo_search\duckduckgo_search.py", line    
         593, in news                                                                                              
             vqd = self._get_vqd(keywords)                                                                         
                   ^^^^^^^^^^^^^^^^^^^^^^^                                                                         
           File "e:\agentic_ai_rag_system\.venv\Lib\site-packages\duckduckgo_search\duckduckgo_search.py", line    
         129, in _get_vqd                                                                                          
             resp_content = self._get_url("GET", "https://duckduckgo.com", params={"q": keywords})                 
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                 
           File "e:\agentic_ai_rag_system\.venv\Lib\site-packages\duckduckgo_search\duckduckgo_search.py", line    
         125, in _get_url                                                                                          
             raise DuckDuckGoSearchException(f"{resp.url} return None. {params=} {content=} {data=}")              
         duckduckgo_search.exceptions.DuckDuckGoSearchException:                                                   
         https://duckduckgo.com?q=shipping+news+from+Chittagong+Port return None. params={'q': 'shipping news from 
         Chittagong Port'} content=None data=None

WARNING  Error reading article from https://www.searates.com/distance-time/: Article `download()` failed with      
         Website protected with Cloudflare, url: None on URL https://www.searates.com/distance-time/

WARNING  Error reading article from https://www.fluentcargo.com/routes/chittagong-bd/new-york-us: Article is binary
         data: https://www.fluentcargo.com/routes/chittagong-bd/new-york-us

Output()

Output()

{'query': 'Test query: Current shipping rates from Chittagong Port and how much time it can take to reach New York?', 'timestamp': '2025-02-20 20:01:26', 'classification': ['hybrid'], 'retrieved_data': {'web_news': None, 'rag': None}, 'final_response': {'result': None}, 'status': 'success'}


In [15]:
from enum import Enum
from datetime import datetime
from dotenv import load_dotenv
import os
from textwrap import dedent

# Placeholder imports (replace with actual Agenta and Groq imports)
# Assuming Agenta provides an Agent class; adjust based on actual API
try:
    from agenta import Agent  # Hypothetical import; replace with correct Agenta module
    from groq import Groq     # Replace with actual Groq client library
except ImportError:
    # Mock classes for demonstration
    class Groq:
        def __init__(self, id):
            self.id = id

    class Agent:
        def __init__(self, model, tools=None, description="", instructions="", markdown=True, show_tool_calls=False):
            self.model = model
            self.tools = tools or []
            self.description = description
            self.instructions = instructions
            self.markdown = markdown
            self.show_tool_calls = show_tool_calls
        
        def print_response(self, prompt, stream=False):
            # Mock response for Llama 3 70B via Groq
            query = prompt.split('Query: "')[1].split('"')[0].lower()
            if all(kw in query for kw in ["inventory", "transportation"]) and not any(kw in query for kw in ["current", "latest", "news", "update"]):
                return "RAG"
            elif any(kw in query for kw in ["current", "latest", "news", "update"]) and not any(kw in query for kw in ["inventory", "transportation"]):
                return "WEB_NEWS"
            else:
                return "HYBRID"

# Mock tools (replace with actual Agenta-compatible tools if available)
class DuckDuckGoTools:
    pass

class QueryType(Enum):
    WEB_NEWS = "web_news"          # Web/News Agent only
    RAG = "rag"                    # RAG only for internal data
    HYBRID = "hybrid"              # Both RAG and Web/News

class RouterAgent:
    def __init__(self, debug: bool = False):
        """Initialize Router Agent with Agenta and Groq Llama 3 70B"""
        self.debug = debug
        load_dotenv()
        
        if not os.getenv('GROQ_API_KEY'):
            raise ValueError("GROQ_API_KEY environment variable not set")
        
        # Define the router using Agenta Agent with Groq Llama 3 70B
        self.router = Agent(
            model=Groq(id="llama3-70b-8192"),
            tools=[DuckDuckGoTools()],  # Add tools as per Agenta’s structure
            description=dedent("""\
                Router Agent for classifying supply chain queries into RAG, Web/News, or Hybrid pathways.
                Uses company-specific inventory and transportation data in RAG and external web data.
            """),
            instructions=dedent("""\
                Analyze queries and classify them based on data needs using the provided prompt.
            """),
            markdown=True,
            show_tool_calls=True  # Enable tool call visibility as requested
        )

    def _log(self, message: str) -> None:
        if self.debug:
            current_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
            print(f"[{current_time}] [DEBUG] {message}")

    def classify_query(self, query: str) -> QueryType:
        """Classify query using Agenta Agent with Llama 3 70B"""
        self._log(f"Classifying query: {query}")
        
        # Optimized prompt for classification
        classification_prompt = dedent(f"""\
            You are an expert router for supply chain queries, leveraging company-specific data and external sources.
            Analyze the following query and classify it into ONE of three pathways based on its data requirements:
            
            Query: "{query}"
            
            Context:
            - RAG contains all internal company data:
              - Inventory: Stock levels, warehouse details, product availability, etc.
              - Transportation: Shipping schedules, logistics, routes, fleet details, etc.
            - Web/News Search Agent provides current external data: news, market updates, web-sourced information.
            
            Pathways:
            1. RAG:
               - Use this if the query can be fully answered with internal inventory and transportation data from RAG.
               - Examples: "What’s the current stock of Product X?", "Which trucks are scheduled for tomorrow?"
               - Key Indicators: Focuses solely on internal data; no external news or updates needed.
            
            2. WEB_NEWS:
               - Use this if the query requires only current external data (e.g., news, updates, web searches) and does not need internal RAG data.
               - Examples: "What’s the latest shipping industry news?", "Current global shipping rates?"
               - Key Indicators: Time-sensitive external data only; no internal inventory/transportation reference.
            
            3. HYBRID:
               - Use this if the query requires BOTH internal RAG data (inventory/transportation) AND current external web/news data.
               - Examples: "How do current shipping delays impact our inventory?", "Combine latest fuel prices with our transportation costs."
               - Key Indicators: Needs synthesis of internal data and external updates.
            
            Instructions:
            1. Determine the query’s intent and scope:
               - Is it asking for internal company data, external updates, or both?
            2. Evaluate data sources needed:
               - RAG only: Internal inventory/transportation suffices.
               - WEB_NEWS only: External current data suffices.
               - HYBRID: Both internal and external data required.
            3. Classify into ONE pathway: RAG, WEB_NEWS, or HYBRID.
            4. Return ONLY the category name (e.g., "RAG", "WEB_NEWS", "HYBRID") with no additional text.
            5. Default to HYBRID if the query’s needs are ambiguous or overlap significantly.
            
            Ensure precise classification for efficient query handling.
        """)
        
        # Call Agenta Agent with Llama 3 70B (simulated here)
        response = self.router.print_response(classification_prompt, stream=False)
        
        try:
            query_type = QueryType(response.lower())
            self._log(f"Classified as: {query_type.value}")
            return query_type
        except ValueError:
            self._log(f"Invalid category received: {response}; defaulting to HYBRID")
            return QueryType.HYBRID

    def route_query(self, query: str) -> dict:
        """Route query and print where data is sent"""
        timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        query_type = self.classify_query(query)

        self._log(f"Processing {query_type.value}")
        
        if query_type == QueryType.WEB_NEWS:
            print("Data sent to Web/News Search Agent")
            result = "Simulated Web/News data retrieval"
        elif query_type == QueryType.RAG:
            print("Data sent to RAG")
            result = "Simulated RAG response based on internal inventory and transportation data"
        elif query_type == QueryType.HYBRID:
            print("Data sent to all agents and RAG")
            result = "Simulated combined response from Web/News and RAG"

        return {
            "query": query,
            "timestamp": timestamp,
            "classification": query_type.value,
            "final_response": {"result": result},
            "status": "success"
        }

if __name__ == "__main__":
    agent = RouterAgent(debug=True)
    queries = [
        "What’s the latest update for Los Angeles port?",
        "Combine latest fuel prices with our transportation costs.",
        "How do current shipping delays in China Port impact our inventory?"
    ]
    
    for query in queries:
        print(f"\nRouting query: {query}")
        result = agent.route_query(query)
        print(result)


Routing query: What’s the latest update for Los Angeles port?
[2025-02-20 20:25:17] [DEBUG] Classifying query: What’s the latest update for Los Angeles port?
[2025-02-20 20:25:17] [DEBUG] Classified as: web_news
[2025-02-20 20:25:17] [DEBUG] Processing web_news
Data sent to Web/News Search Agent
{'query': 'What’s the latest update for Los Angeles port?', 'timestamp': '2025-02-20 20:25:17', 'classification': 'web_news', 'final_response': {'result': 'Simulated Web/News data retrieval'}, 'status': 'success'}

Routing query: Combine latest fuel prices with our transportation costs.
[2025-02-20 20:25:17] [DEBUG] Classifying query: Combine latest fuel prices with our transportation costs.
[2025-02-20 20:25:17] [DEBUG] Classified as: hybrid
[2025-02-20 20:25:17] [DEBUG] Processing hybrid
Data sent to all agents and RAG
{'query': 'Combine latest fuel prices with our transportation costs.', 'timestamp': '2025-02-20 20:25:17', 'classification': 'hybrid', 'final_response': {'result': 'Simulated 